In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from collections import deque
from sklearn.preprocessing import StandardScaler

In [ ]:
data_df = pd.read_excel('Portfolio_Rebalancing_Data(high).xlsx')

In [ ]:
data_df.info()

In [ ]:
data_df.head()

In [ ]:
data_df.isnull.sum()

In [ ]:
plt.figure(figsize=(13,5))
plt.plot(data_df['Date'],data_df['NIFTY50_Price'],linestyle='-',color='b')

plt.figure(figsize=(13,5))
plt.plot(data_df['Date'],data_df['GSEC_10Y_Price'],linestyle='-',color='r')

In [ ]:
EPISODES = 100
GAMMA = 0.99
LEARNING_RATE = 0.001
MEMORY_SIZE = 10000
BATCH_SIZE = 64
TRAINING_START = 500
INITIAL_BALANCE = 100000

EPSILON_MAX = 1.0
EPSILON_MIN = 0.01
EPSILON_DECAY_RATE = 0.995

ACTION_SPACE_SIZE = 3
ASSET_COUNT = 3 # Stocks, Bonds, Cash
STATE_SIZE = ASSET_COUNT + 2 # Weights (3) + Scaled Returns (2)


data_df['Stocks_Return'] = data_df['NIFTY50_Price'].pct_change().fillna(0)
data_df['Bonds_Return'] = data_df['GSEC_10Y_Price'].pct_change().fillna(0)
data_df['Cash_Return'] = data_df['Fixed_Deposit_Rate'] / 25200 # Daily approx

market_returns = data_df[['Stocks_Return', 'Bonds_Return', 'Cash_Return']].values
market_features_raw = data_df[['Stocks_Return', 'Bonds_Return']].values

scaler = StandardScaler()
market_features_scaled = scaler.fit_transform(market_features_raw)

TOTAL_STEPS = len(data_df) - 1


weights = np.array([1/3, 1/3, 1/3])
balance = INITIAL_BALANCE
current_step = 0

initial_market_state = market_features_scaled[current_step]
state = np.concatenate([weights, initial_market_state])

memory = deque(maxlen=MEMORY_SIZE)

model = keras.Sequential([
    keras.layers.Input(shape=(STATE_SIZE,)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(ACTION_SPACE_SIZE, activation='linear')
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='mse')

target_model = keras.Sequential([
    keras.layers.Input(shape=(STATE_SIZE,)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(ACTION_SPACE_SIZE, activation='linear')
])
target_model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='mse')
target_model.set_weights(model.get_weights())

In [ ]:
# Training Loop
for episode in range(EPISODES):

    weights = np.array([1/3, 1/3, 1/3])
    balance = INITIAL_BALANCE
    current_step = 0
    initial_market_state = market_features_scaled[current_step]
    state = np.concatenate([weights, initial_market_state])
    done = False
    total_reward = 0
    step = 0

    while not done:


        if np.random.rand() <= epsilon:
            action = np.random.randint(ACTION_SPACE_SIZE)
        else:
            q_values = model.predict(state.reshape(1, STATE_SIZE), verbose=0)
            action = np.argmax(q_values[0])

        rebalance_size = 0.05

        if action == 0:
            stocks_change = min(rebalance_size, 0.7 - weights[0])
            bonds_change = -min(rebalance_size, weights[1])
            cash_change = -(stocks_change + bonds_change)
            weights += np.array([stocks_change, bonds_change, cash_change])

        elif action == 1:
            stocks_change = -min(rebalance_size, weights[0])
            bonds_change = min(rebalance_size, 0.7 - weights[1])
            cash_change = -(stocks_change + bonds_change)
            weights += np.array([stocks_change, bonds_change, cash_change])


        weights = np.maximum(0, weights / np.sum(weights))


        current_step += 1
        done = current_step >= TOTAL_STEPS

        current_returns = market_returns[current_step]
        portfolio_return = np.dot(weights, current_returns)

        new_balance = balance * (1 + portfolio_return)

        penalty = 0.0001 if action != 2 else 0
        reward = np.log(new_balance) - np.log(balance) - penalty
        balance = new_balance

        next_market_state = market_features_scaled[current_step]
        next_state = np.concatenate([weights, next_market_state])

        memory.append((state, action, reward, next_state, done))
        state = next_state
        total_reward += reward
        step += 1

        if len(memory) > TRAINING_START:
            if len(memory) >= BATCH_SIZE:


                batch_indices = np.random.choice(len(memory), BATCH_SIZE, replace=False)
                batch = [memory[i] for i in batch_indices]


                states_b = np.array([x[0] for x in batch])
                actions_b = np.array([x[1] for x in batch])
                rewards_b = np.array([x[2] for x in batch])
                next_states_b = np.array([x[3] for x in batch])
                dones_b = np.array([x[4] for x in batch])

                q_next = target_model.predict(next_states_b, verbose=0)
                targets_q = rewards_b + GAMMA * np.amax(q_next, axis=1) * (1 - dones_b)

                current_q_values = model.predict(states_b, verbose=0)

                for i in range(BATCH_SIZE):
                    current_q_values[i, actions_b[i]] = targets_q[i]

                model.fit(states_b, current_q_values, epochs=1, verbose=0)

    epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY_RATE)

    target_model.set_weights(model.get_weights())

    final_balance = balance
    print(f"Episode: {episode + 1}/{EPISODES}, Steps: {step}, Epsilon: {epsilon:.4f}, "
          f"Total Log-Reward: {total_reward:.4f}, Final Balance: ${final_balance:,.2f}")

Episode: 1/100, Steps: 364, Epsilon: 0.9950, Total Log-Reward: -0.2864, Final Balance: $76,929.07
Episode: 2/100, Steps: 364, Epsilon: 0.9900, Total Log-Reward: -0.2953, Final Balance: $76,385.89
Episode: 3/100, Steps: 364, Epsilon: 0.9851, Total Log-Reward: -0.0783, Final Balance: $94,773.40
Episode: 4/100, Steps: 364, Epsilon: 0.9801, Total Log-Reward: -0.2121, Final Balance: $82,925.23
Episode: 5/100, Steps: 364, Epsilon: 0.9752, Total Log-Reward: -0.1936, Final Balance: $84,453.16
Episode: 6/100, Steps: 364, Epsilon: 0.9704, Total Log-Reward: -0.0725, Final Balance: $95,290.11
Episode: 7/100, Steps: 364, Epsilon: 0.9655, Total Log-Reward: -0.1230, Final Balance: $90,621.16
Episode: 8/100, Steps: 364, Epsilon: 0.9607, Total Log-Reward: -0.2265, Final Balance: $81,647.94
Episode: 9/100, Steps: 364, Epsilon: 0.9559, Total Log-Reward: -0.1995, Final Balance: $83,962.86
Episode: 10/100, Steps: 364, Epsilon: 0.9511, Total Log-Reward: 0.0376, Final Balance: $106,363.04
Episode: 11/100, St

In [ ]:
print("\n--- Final Evaluation ---")


weights = np.array([1/3, 1/3, 1/3])
balance = INITIAL_BALANCE
current_step = 0
initial_market_state = market_features_scaled[current_step]
test_state = np.concatenate([weights, initial_market_state])
test_done = False
test_balance_history = [balance]

while not test_done:
    q_values = model.predict(test_state.reshape(1, STATE_SIZE), verbose=0)
    test_action = np.argmax(q_values[0])

    rebalance_size = 0.05

    if test_action == 0:
        stocks_change = min(rebalance_size, 0.7 - weights[0])
        bonds_change = -min(rebalance_size, weights[1])
        cash_change = -(stocks_change + bonds_change)
        weights += np.array([stocks_change, bonds_change, cash_change])
    elif test_action == 1:
        stocks_change = -min(rebalance_size, weights[0])
        bonds_change = min(rebalance_size, 0.7 - weights[1])
        cash_change = -(stocks_change + bonds_change)
        weights += np.array([stocks_change, bonds_change, cash_change])

    weights = np.maximum(0, weights / np.sum(weights))

    current_step += 1
    test_done = current_step >= TOTAL_STEPS
    current_returns = market_returns[current_step]
    portfolio_return = np.dot(weights, current_returns)
    balance = balance * (1 + portfolio_return)

    if not test_done:
        next_market_state = market_features_scaled[current_step]
        test_state = np.concatenate([weights, next_market_state])

    test_balance_history.append(balance)

test_returns = pd.Series(test_balance_history).pct_change().dropna()

# Sharpe Ratio
annualizing_factor = np.sqrt(252)
sharpe_ratio = np.mean(test_returns) / np.std(test_returns) * annualizing_factor

# Maximum Drawdown
wealth_index = pd.Series(test_balance_history)
peak = wealth_index.cummax()
drawdown = (wealth_index - peak) / peak
max_drawdown = drawdown.min() * 100

print(f"Final Test Sharpe Ratio: {sharpe_ratio:.4f}")
print(f"Final Test Max Drawdown: {max_drawdown:.2f}%")
print("\nSuccess Metrics Check:")
print(f"- Target: Higher Sharpe ratio than benchmarks (Achieved: {sharpe_ratio:.4f})")
print(f"- Target: Reduce maximum drawdown by 20% (Achieved: {max_drawdown:.2f}%)")


--- Final Evaluation ---
Final Test Sharpe Ratio: 0.7972
Final Test Max Drawdown: -7.48%

Success Metrics Check:
- Target: Higher Sharpe ratio than benchmarks (Achieved: 0.7972)
- Target: Reduce maximum drawdown by 20% (Achieved: -7.48%)


In [ ]:
#checking on other data sets
STATE_SIZE = 5
ACTION_SPACE_SIZE = 3
ASSET_COUNT = 3
INITIAL_BALANCE = 100000

REBALANCE_SIZE = 0.05



try:
    test_data_df = pd.read_csv('medium volitility.csv')
except FileNotFoundError:
    print("Error: market_data_test.csv not found. Creating dummy test data.")
    # Create dummy data for a runnable example
    dates = pd.date_range(start='2024-01-01', periods=252, freq='B') # ~1 year
    np.random.seed(99)
    data = {
        'Date': dates,
        'NIFTY_Price': 20000 + np.cumsum(np.random.normal(0, 70, 252)),
        'GOVSEC': 7 + np.cumsum(np.random.normal(0, 0.005, 252)),
        'Fixed_Deposit_Rate': 8 + np.random.normal(0, 0.05, 252)
    }
    test_data_df = pd.DataFrame(data)

test_data_df['Stocks_Return'] = test_data_df['NIFTY_Price'].pct_change().fillna(0)
test_data_df['Bonds_Return'] = test_data_df['GOVSEC'].pct_change().fillna(0)
test_data_df['Cash_Return'] = test_data_df['Fixed_Deposit_Rate'] / 25200

market_returns_test = test_data_df[['Stocks_Return', 'Bonds_Return', 'Cash_Return']].values
market_features_raw_test = test_data_df[['Stocks_Return', 'Bonds_Return']].values

test_scaler = StandardScaler()
market_features_scaled_test = test_scaler.fit_transform(market_features_raw_test)

TOTAL_TEST_STEPS = len(test_data_df) - 1

model = keras.Sequential([
    keras.layers.Input(shape=(STATE_SIZE,)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(ACTION_SPACE_SIZE, activation='linear')
])

try:
    model.load_weights('portfolio_agent_weights.h5')
    print("Successfully loaded trained model weights.")
except:
    print("Warning: Could not load weights. Model will run with initial random weights. Please save weights during training.")


print("\n--- Starting Evaluation Simulation ---")


weights = np.array([1/3, 1/3, 1/3])
balance = INITIAL_BALANCE
current_step = 0
initial_market_state = market_features_scaled_test[current_step]
test_state = np.concatenate([weights, initial_market_state])
test_done = False
test_balance_history = [balance]

while not test_done:


    q_values = model.predict(test_state.reshape(1, STATE_SIZE), verbose=0)
    test_action = np.argmax(q_values[0])


    if test_action == 0:
        stocks_change = min(REBALANCE_SIZE, 0.7 - weights[0])
        bonds_change = -min(REBALANCE_SIZE, weights[1])
        cash_change = -(stocks_change + bonds_change)
        weights += np.array([stocks_change, bonds_change, cash_change])
    elif test_action == 1:
        stocks_change = -min(REBALANCE_SIZE, weights[0])
        bonds_change = min(REBALANCE_SIZE, 0.7 - weights[1])
        cash_change = -(stocks_change + bonds_change)
        weights += np.array([stocks_change, bonds_change, cash_change])

    weights = np.maximum(0, weights / np.sum(weights))
    current_step += 1
    test_done = current_step >= TOTAL_TEST_STEPS

    if not test_done:
        current_returns = market_returns_test[current_step]
        portfolio_return = np.dot(weights, current_returns)
        balance = balance * (1 + portfolio_return)


        next_market_state = market_features_scaled_test[current_step]
        test_state = np.concatenate([weights, next_market_state])

    test_balance_history.append(balance)


test_returns = pd.Series(test_balance_history).pct_change().dropna()


annualizing_factor = np.sqrt(252)


sharpe_ratio = np.mean(test_returns) / np.std(test_returns) * annualizing_factor if np.std(test_returns) != 0 else 0

wealth_index = pd.Series(test_balance_history)
peak = wealth_index.cummax()
drawdown = (wealth_index - peak) / peak
max_drawdown = drawdown.min() * 100

print("\n--- Evaluation Results ---")
print(f"Final Portfolio Value: ${balance:,.2f}")
print(f"Total Return: {(balance / INITIAL_BALANCE - 1) * 100:.2f}%")
print(f"Test Period Sharpe Ratio (Annualized): {sharpe_ratio:.4f}")
print(f"Test Period Maximum Drawdown: {max_drawdown:.2f}%")
print("\nCompare these metrics against your defined benchmarks to assess efficiency.")

Error: market_data_test.csv not found. Creating dummy test data.

--- Starting Evaluation Simulation ---

--- Evaluation Results ---
Final Portfolio Value: $103,817.40
Total Return: 3.82%
Test Period Sharpe Ratio (Annualized): 4.6117
Test Period Maximum Drawdown: -0.20%

Compare these metrics against your defined benchmarks to assess efficiency.
